# CNN tier: off-the-shelf vs fine-tuned

Reads `results/runs.jsonl` — the committed record — and plots it. No pipeline logic
lives here; producing rows is `cbir evaluate`'s job.

Companion to `02_classic_comparison.ipynb`, which covers BoW/VLAD/Fisher. Styling and
palettes are shared via `style.py`; colour is assigned by the job it does and every
palette was checked with a validator rather than by eye.

To add points, from the repo root:

```bash
cbir evaluate-preset gem-r101
cbir evaluate-preset gem-ft-r101
cbir evaluate-preset rmac-r101
```

In [ ]:
import sys

import matplotlib.pyplot as plt
import pandas as pd

sys.path.insert(0, ".")  # style.py sits beside this notebook
import style
from cbir.eval.frame import tidy_rows
from cbir.eval.results import latest

style.use()
PROTOCOLS = style.PROTOCOLS

## The data

`latest()` collapses re-runs of a configuration to the most recent row, so a re-measured
point replaces rather than duplicates. `tier` is derived rather than recorded: it is the
axis this project's story runs along, and it follows from `weights` plus the technique.

This table is also the **accessible view** of every figure below — nothing is encoded by
colour alone.

In [ ]:
rows = tidy_rows(records=list(latest().values()))
df = pd.DataFrame(rows)
df = df[df["technique"].isin(["neural_codes", "gem", "rmac"])].copy()

df["tier"] = [style.tier_of(t, p) for t, p in zip(df["technique"], df["params"])]
for field in ("backbone", "weights", "p", "levels", "whiten", "whiten_source", "last_pool"):
    df[field] = [p.get(field) for p in df["params"]]
# Recorded before `weights` existed means it ran on the default.
df["weights"] = df["weights"].fillna("torchvision")
df["scales"] = [len(p.get("scales") or [1.0]) for p in df["params"]]

print(f"{df['commit'].nunique()} commit(s), {len(df) // 3} run(s)")
df[["technique", "backbone", "weights", "p", "scales", "dim", "protocol", "map", "tier", "seconds"]].head(20)

## mAP against descriptor width, by tier

The chart the whole project builds toward. `dim` is the encoded vector length — the axis
on which eras are actually comparable, since a classic descriptor buys its accuracy with
length and a CNN one does not.

Colour is an **ordinal** ramp, not three unrelated hues: the tiers are an ordered
sequence and darker reads as later. Marker shape carries the technique, so identity never
rests on colour alone.

In [ ]:
MARKER = {"neural_codes": "s", "gem": "o", "rmac": "^"}

def tier_scatter(df, protocol="medium", classic=None, ax=None):
    subset = df[(df["protocol"] == protocol)].dropna(subset=["dim", "map"])
    if ax is None:
        _, ax = plt.subplots(figsize=(7.5, 4.2))

    for tier in style.TIER_ORDER:
        rows = subset[subset["tier"] == tier]
        if rows.empty and tier != "classic":
            continue
        for technique, marks in rows.groupby("technique", observed=True):
            ax.scatter(marks["dim"], marks["map"] * 100, s=46, color=style.TIER[tier],
                       marker=MARKER.get(technique, "o"), linewidth=0, alpha=0.85, zorder=3)
        if not rows.empty:
            best = rows.loc[rows["map"].idxmax()]
            style.label_point(ax, best["dim"], best["map"] * 100, f"{tier} best {best['map'] * 100:.1f}")

    # The classic tier is the comparison this chart exists to make, so it is drawn here
    # even though it is another notebook's subject.
    if classic is not None and not classic.empty:
        pts = classic[classic["protocol"] == protocol].dropna(subset=["dim", "map"])
        ax.scatter(pts["dim"], pts["map"] * 100, s=46, color=style.TIER["classic"],
                   marker="D", linewidth=0, alpha=0.85, zorder=3)
        best = pts.loc[pts["map"].idxmax()]
        style.label_point(ax, best["dim"], best["map"] * 100, f"classic best {best['map'] * 100:.1f}",
                          dx=-8, ha="right")

    ax.set_xscale("log")
    ax.set_ylim(bottom=0)
    style.style(ax, title=f"mAP vs descriptor width — roxford5k {protocol.capitalize()}",
                xlabel="descriptor width (log)", ylabel="mAP")

    handles = [plt.Line2D([], [], marker="o", linestyle="", markersize=8,
                          color=style.TIER[t], label=t) for t in style.TIER_ORDER]
    handles += [plt.Line2D([], [], marker=m, linestyle="", markersize=8, color=style.MUTED,
                           label=style.LABEL[t]) for t, m in MARKER.items()]
    handles += [plt.Line2D([], [], marker="D", linestyle="", markersize=8, color=style.MUTED,
                           label="BoW/VLAD/Fisher")]
    # Upper left: the low-width/high-mAP corner is the one region with no marks in it.
    ax.legend(handles=handles, frameon=False, fontsize=8, loc="upper left", ncol=2)
    return ax.figure


classic = pd.DataFrame([r for r in rows if r["technique"] in ("bow", "vlad", "fisher")])
fig_tiers = tier_scatter(df, "medium", classic=classic)
fig_tiers.tight_layout()

## Weight provenance

Same architecture, same dataset, same task — different training recipe. `cnnimageretrieval`
fills a torchvision architecture with **Caffe-converted** ImageNet weights; we started with
torchvision's, and the descriptors agree at cosine 0.23.

The effect is architecture-specific, which is the point: it is not a blanket improvement,
so a table of backbones measured on the wrong weights ranks them wrongly.

In [ ]:
prov = df[(df["technique"] == "gem") & (df["p"] == 3.0) & (df["protocol"] == "medium")
          & (df["whiten"] == True) & (df["scales"] == 3)  # noqa: E712
          & (df["whiten_source"].isin([None, "held_out"]))]
prov = prov[prov["last_pool"].isin([None, True])]
pivot = prov.pivot_table(index="backbone", columns="weights", values="map", aggfunc="max") * 100
pivot = pivot.dropna().sort_values("caffe")

fig, ax = plt.subplots(figsize=(7, 3.6))
positions = range(len(pivot))
width = 0.38
for i, source in enumerate(["torchvision", "caffe"]):
    offset = (i - 0.5) * width
    bars = ax.bar([p + offset for p in positions], pivot[source], width=width * 0.92,
                  color=style.WEIGHTS[source], label=source, zorder=3)
    for bar, value in zip(bars, pivot[source]):
        ax.annotate(f"{value:.1f}", (bar.get_x() + bar.get_width() / 2, value),
                    textcoords="offset points", xytext=(0, 3), ha="center",
                    fontsize=8, color=style.MUTED)

ax.set_xticks(list(positions), list(pivot.index))
ax.legend(frameon=False, fontsize=9, loc="upper left")
style.style(ax, title="GeM p=3, multi-scale, whitened — roxford5k Medium", ylabel="mAP")
# Headroom so the legend clears the tallest bar's value label rather than covering it.
ax.set_ylim(0, float(pivot.to_numpy().max()) * 1.32)
fig.tight_layout()
pivot.round(1)

## Off-the-shelf against fine-tuned, and both against the published rows

The published figures are Radenović et al. ([1803.11285](https://arxiv.org/abs/1803.11285),
Table 5). They are drawn as a **reference tick**, not a third bar: they are a target, not
another measurement of ours.

Note the paper does not evaluate the Easy protocol at all, so it appears here without a
reference mark.

In [ ]:
# (technique-label, backbone) -> {protocol: published mAP}
PUBLISHED = {
    ("off-the-shelf", "vgg16"): {"medium": 40.5, "hard": 15.7},
    ("off-the-shelf", "resnet101"): {"medium": 45.0, "hard": 17.7},
    ("fine-tuned", "vgg16"): {"medium": 61.9, "hard": 33.7},
    ("fine-tuned", "resnet101"): {"medium": 64.7, "hard": 38.5},
}

def ours(backbone, finetuned):
    rows = df[(df["technique"] == "gem") & (df["backbone"] == backbone) & (df["scales"] == 3)
              & (df["whiten"] == True)]  # noqa: E712
    rows = rows[rows["weights"].eq("sfm120k") == finetuned]
    if not finetuned:
        # Pinned to the held-out corpus: `sfm30k` is a different experiment, and a max
        # across both put one protocol's bar on one corpus and another's on the other.
        rows = rows[rows["weights"].eq("caffe") & rows["whiten_source"].isin([None, "held_out"])]
    return {p: rows[rows["protocol"] == p]["map"].max() * 100 for p in PROTOCOLS}

groups = [(kind, backbone) for kind in ("off-the-shelf", "fine-tuned")
          for backbone in ("vgg16", "resnet101")]

fig, axes = plt.subplots(1, 3, figsize=(11.5, 3.8), sharey=True)
for ax, protocol in zip(axes, PROTOCOLS):
    for i, (kind, backbone) in enumerate(groups):
        value = ours(backbone, kind == "fine-tuned")[protocol]
        colour = style.TIER["fine-tuned CNN" if kind == "fine-tuned" else "off-the-shelf CNN"]
        bar = ax.bar([i], [value], width=0.68, color=colour, zorder=3)[0]
        ax.annotate(f"{value:.1f}", (bar.get_x() + bar.get_width() / 2, value),
                    textcoords="offset points", xytext=(0, 3), ha="center",
                    fontsize=8, color=style.MUTED)
        reference = PUBLISHED[(kind, backbone)].get(protocol)
        if reference is not None:
            ax.plot([i - 0.36, i + 0.36], [reference, reference], color=style.INK,
                    linewidth=1.6, zorder=4)
    ax.set_xticks(range(len(groups)), [f"{b}\n{k}" for k, b in groups], fontsize=8)
    style.style(ax, title=protocol.capitalize())

axes[0].set_ylabel("mAP")
axes[0].set_ylim(bottom=0)
handles = [plt.Line2D([], [], marker="s", linestyle="", markersize=9,
                      color=style.TIER["off-the-shelf CNN"], label="off-the-shelf"),
           plt.Line2D([], [], marker="s", linestyle="", markersize=9,
                      color=style.TIER["fine-tuned CNN"], label="fine-tuned"),
           plt.Line2D([], [], linestyle="-", color=style.INK, linewidth=1.6, label="published")]
axes[-1].legend(handles=handles, frameon=False, fontsize=8, loc="upper left")
fig_validation = fig
fig.tight_layout()

## The pooling exponent

GeM's `p` interpolates between average pooling (`p=1`, SPoC) and max (`p→∞`, MAC). The
curve is flat across its top on Medium — 3, 4 and 5 sit inside 0.8 mAP, which one seed
cannot resolve — while Hard keeps rising toward true max.

Which makes the fine-tuned checkpoints' trained value the interesting number: `p` is
differentiable and so was *learned*, and it came out at 2.92, three percent from the
hand-picked 3.0 — marked here as the dotted rule.

Unlike the bar charts above, **this y-axis does not start at zero**: the subject is the
shape of the curve, not the magnitude of each point. Read the vertical extent as
"differences between exponents", never as "how good GeM is".

In [ ]:
TRAINED_P = 2.92  # what the fine-tuned vgg16 checkpoint learned; see finetuned.py

sweep = df[(df["technique"] == "gem") & (df["backbone"] == "alexnet")
           & (df["whiten"] == False) & (df["dim"].isna() | df["dim"].eq(256))  # noqa: E712
           & (df["scales"] == 3)]

fig, axes = plt.subplots(1, 2, figsize=(9, 3.4))
for ax, protocol in zip(axes, ["medium", "hard"]):
    rows = sweep[sweep["protocol"] == protocol]
    finite = rows[rows["p"].notna()].sort_values("p")
    ax.plot(finite["p"], finite["map"] * 100, color=style.TIER["off-the-shelf CNN"],
            linewidth=2, marker="o", markersize=6, zorder=3)

    mac = rows[rows["p"].isna()]
    if not mac.empty:
        value = mac["map"].max() * 100
        ax.axhline(value, color=style.MUTED, linewidth=1.2, linestyle=(0, (4, 3)), zorder=2)
        # Offset above the rule; printed on it, the text and the dashes overlapped.
        style.label_point(ax, finite["p"].max(), value, f"MAC (p→∞) {value:.1f}",
                          dx=0, dy=7, ha="right")

    best = finite.loc[finite["map"].idxmax()]
    style.label_point(ax, best["p"], best["map"] * 100, f"{best['map'] * 100:.1f} at p={best['p']:.0f}")
    ax.axvline(TRAINED_P, color=style.TIER["fine-tuned CNN"], linewidth=1.2, linestyle=":", zorder=2)
    style.label_point(ax, TRAINED_P, ax.get_ylim()[0], f"trained p={TRAINED_P}", dx=4, dy=10,
                      rotation=90, va="bottom")
    style.style(ax, title=protocol.capitalize(), xlabel="pooling exponent p", ylabel="mAP")

axes[0].set_ylabel("mAP")
fig.tight_layout()

## Write the README figures

In [ ]:
from pathlib import Path

DOCS = Path("..") / "docs"
DOCS.mkdir(exist_ok=True)

# White rather than transparent: GitHub renders READMEs on a dark background too, and a
# transparent PNG would leave the dark ink invisible there.
for name, figure in (("cnn_tiers", fig_tiers), ("cnn_validation", fig_validation)):
    path = DOCS / f"{name}.png"
    figure.savefig(path, dpi=110, bbox_inches="tight", facecolor="white")
    print("wrote", path.resolve())